# Chemical equilibrium and Le Chatelier

*physchem · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/physchem/chemical_equilibrium)

Where does a reaction stop? From $\Delta G^\circ = -RT\ln K$ the equilibrium extent $\xi$ of a gas reaction ($\mathrm{N_2O_4 \rightleftharpoons 2\,NO_2}$ or the Haber reaction $\mathrm{N_2 + 3H_2 \rightleftharpoons 2NH_3}$) is found for hundreds of temperatures and pressures at once by bisection on $Q(\xi) = K$, showing the Le Chatelier shifts with pressure, temperature (van 't Hoff) and added product.

**Vector form**

$$\Delta G^\circ = \Delta H^\circ - T\Delta S^\circ = -RT\ln K,\qquad Q = \prod_i a_i^{\nu_i} = K,\qquad a_i = \frac{n_i}{n}\,\frac{P}{P^\circ},\qquad \frac{d\ln K}{d(1/T)} = -\frac{\Delta H^\circ}{R}$$

**Index (tensor) form**

$$n_{bi} = n^0_{bi} + \nu_i\,\xi_b \qquad \ln Q_b = \nu_i \ln\!\Big(\frac{n_{bi}}{n_{bj}\mathbb{1}_j}\,\frac{P_b}{P^\circ}\Big) \qquad \ln Q_b(\xi_b) = \ln K_b = -\frac{\Delta H^\circ - T_b\Delta S^\circ}{R\,T_b}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\nu_i$ | `r['nu']` | (S,) | stoichiometric numbers: $-1, 2$ for $\mathrm{N_2O_4 \rightleftharpoons 2NO_2}$; $-1, -3, 2$ for ammonia |
| $T_b,\ P_b$ | `condition_grid(T_list, P_grid)` | (B,) | every (temperature, pressure) pair flattened into one batch index $b$ |
| $\ln K_b$ | `log_K(T_b, dH, dS)` | (B,) | $-\Delta G^\circ/RT$ with approximate, temperature-independent $\Delta H^\circ$, $\Delta S^\circ$ |
| $n_{bi} = n^0_{bi} + \nu_i\xi_b$ | `n0 + einsum('i,b->bi', nu, xi)` | (B, S) | moles of each species; an outer product, nothing summed |
| $n_{bj}\mathbb{1}_j$ | `n.sum(-1, keepdim=True)` | (B, 1) | total moles; $j$ summed |
| $\nu_i \ln a_{bi}$ | `einsum('i,bi->b', nu, ln_a)` | (B,) | $\ln Q$: the species index $i$ is summed |
| $\xi_{\min,b},\ \xi_{\max,b}$ | `extent_bounds(n0, nu)` | (B,) | no amount may become negative: the bracket for bisection |
| $\xi_b$ | `equilibrium_extent(n0, nu, P, lnK)` | (B,) | bisection, all conditions together: keep the half where $\ln Q - \ln K$ changes sign |
| $-\Delta H^\circ / R$ | `van_t_hoff_slope(dH)` | () | slope of $\ln K$ against $1/T$ |

$Q$ measures where the mixture is, $K$ where it wants to be: if $Q < K$ the reaction runs forward, if $Q > K$ it runs back. Because $\ln Q$ rises steadily with the extent $\xi$ (from $-\infty$ with no products to $+\infty$ with no reactants), the equilibrium is the single root of $\ln Q(\xi) - \ln K$, and bisection finds it for every condition at once. Le Chatelier's principle falls out of the formula. Pressure: with ideal-gas activities $Q = K_x\,(P/P^\circ)^{\Delta\nu}$ where $\Delta\nu = \sum_i \nu_i$, so raising $P$ favours the side with fewer gas molecules ($\mathrm{N_2O_4}$ dissociates less, more $\mathrm{NH_3}$ forms). Temperature: $\ln K = -\Delta H^\circ/RT + \Delta S^\circ/R$, so $K$ grows with $T$ for an endothermic reaction ($\mathrm{N_2O_4}$, the brown $\mathrm{NO_2}$ colour deepens on warming) and falls for an exothermic one (ammonia). Added product: $Q$ jumps above $K$ and some product turns back into reactants, but not all of it. For $\mathrm{N_2O_4}$ the result can be checked by hand: $K = 4\alpha^2 P/((1-\alpha^2)P^\circ)$ gives $\alpha = \sqrt{K/(K + 4P/P^\circ)}$. The thermodynamic data are approximate textbook values at 298 K ($\Delta H^\circ \approx +57.2$ kJ/mol, $\Delta S^\circ \approx +176$ J/(mol K) for $\mathrm{N_2O_4}$; $\approx -92.2$ kJ/mol and $\approx -199$ J/(mol K) for ammonia), treated as constant in $T$, and the gases as ideal; both simplifications overestimate the ammonia yield at 700 K and 100 bar somewhat (measured: roughly 20 % $\mathrm{NH_3}$ rather than about 30 %). The Haber reaction is strongly favoured at room temperature but far too slow there: plants run at about 700–750 K and 150–300 bar with an iron catalyst, trading yield for speed. Try T = 700 K with the Haber reaction.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/physchem/chemical_equilibrium/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    r = tensors.reaction(params["reaction"], dev)
    nu, dH, dS = r["nu"], r["dH"], r["dS"]
    T_list = r["T_curves"] + [params["T"]]  # the chosen temperature is the last curve
    P_grid = tensors.pressures(dev)  # (P,) bar
    T_b, P_b = tensors.condition_grid(T_list, P_grid)  # (B,), (B,) with B = nT · P
    n0 = r["n0"].expand(len(T_b), -1)  # (B, S) same feed everywhere

    # 1) equilibrium for every (T, P) at once
    lnK = equation.log_K(T_b, dH, dS)  # (B,)
    xi, history = solve.equilibrium_extent(n0, nu, P_b, lnK)  # (B,)
    n_eq = equation.moles(n0, nu, xi)  # (B, S)
    residual = (equation.log_Q(n_eq, P_b, nu) - lnK).abs().max().item()
    _, xi_max = equation.extent_bounds(n0, nu)
    conversion = (xi / xi_max).reshape(len(T_list), len(P_grid))  # (nT, P) fraction of the possible extent
    x_eq = (n_eq / n_eq.sum(-1, keepdim=True)).reshape(len(T_list), len(P_grid), -1)  # (nT, P, S)

    T0 = params["T"]
    K0 = math.exp(equation.log_K(torch.tensor(T0, dtype=tensors.DTYPE), dH, dS).item())
    print(f"{r['label']}   ΔH° ≈ {dH / 1e3:+.1f} kJ/mol   ΔS° ≈ {dS:+.1f} J/(mol K)   (approximate 298 K values)")
    print(f"T = {T0:.0f} K:  ΔG° = {(dH - T0 * dS) / 1e3:+.2f} kJ/mol   K = {K0:.4g}")
    print(f"bisection on {len(T_b)} conditions: max |ln Q − ln K| after 10/30/60 halvings = "
          f"{history[9]:.1e} / {history[29]:.1e} / {history[-1]:.1e}   (final check {residual:.1e})")  # fmt: skip

    print("\npressure (Le Chatelier): conversion ξ/ξ_max at the chosen T")
    for P in (0.1, 1.0, 10.0, 100.0):
        p = int(torch.argmin((P_grid - P).abs()))
        print(f"  P = {P_grid[p].item():6.1f} bar:  {conversion[-1, p].item():.4f}")

    p1 = int(torch.argmin((P_grid - params["P"]).abs()))
    print(f"\ntemperature (van 't Hoff, d ln K/d(1/T) = −ΔH°/R = {equation.van_t_hoff_slope(dH):.0f} K):"
          f" conversion at P = {P_grid[p1].item():.3g} bar")  # fmt: skip
    for t, T in enumerate(T_list[:-1]):
        print(f"  T = {T:5.0f} K:  K = {math.exp(lnK.reshape(len(T_list), -1)[t, 0].item()):.4g}"
              f"   conversion = {conversion[t, p1].item():.4f}")  # fmt: skip

    # 2) Le Chatelier with added product at the chosen (T, P)
    P1 = torch.tensor([params["P"]], device=dev, dtype=tensors.DTYPE)
    lnK1 = equation.log_K(torch.tensor([T0], device=dev, dtype=tensors.DTYPE), dH, dS)
    xi1, _ = solve.equilibrium_extent(r["n0"][None], nu, P1, lnK1)
    n1 = equation.moles(r["n0"][None], nu, xi1)  # (1, S)
    n_add, gap, n_new = solve.add_product_and_reequilibrate(n1, nu, P1, lnK1, params["add_product"])
    prod = r["species"][-1]
    print(f"\nadd {params['add_product']:.2f} mol {prod} at T = {T0:.0f} K, P = {params['P']:.3g} bar:")
    print("  species   " + "".join(f"{s:>9}" for s in r["species"]))
    for name, n in (("equilibrium", n1), ("just after", n_add), ("new equil.", n_new)):
        print(f"  {name:<11}" + "".join(f"{v:9.4f}" for v in n[0].tolist()) + "  mol")
    shift = n_new[0, -1].item() - n_add[0, -1].item()
    print(f"  Q/K just after adding = {math.exp(gap.item()):.3f}  →  net change {shift:+.4f} mol {prod}"
          " (Q > K: back towards reactants)")  # fmt: skip

    T_line = torch.linspace(min(T_list) - 10, max(T_list) + 10, 100, device=dev, dtype=tensors.DTYPE)
    return plot.animate(
        r, T_list, P_grid, conversion, x_eq, T_line, equation.log_K(T_line, dH, dS), params["P"],
        (n1[0], n_add[0], n_new[0]), out_dir / "chemical_equilibrium.gif",
    )  # fmt: skip


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/physchem/chemical_equilibrium/equation.py
r"""Physics → tensor translation.

A gas reaction  Σ_i ν_i X_i = 0  (ν_i < 0 for reactants, > 0 for products) has equilibrium constant

    ΔG°(T) = ΔH° − T ΔS°,        K(T) = exp(−ΔG° / (R T))

For a batch of conditions b (temperature T_b, pressure P_b, starting moles n⁰_bi) and an extent of reaction ξ_b:

    n_bi = n⁰_bi + ν_i ξ_b                     moles of species i
    a_bi = (n_bi / n_b) (P_b / P°)             ideal-gas activity = partial pressure in bar, n_b = Σ_j n_bj
    ln Q_b = ν_i ln a_bi                        reaction quotient (i summed)

Equilibrium is reached when Q_b = K_b. The sum over species is an einsum; the batch index b is never summed.
"""

import torch

R_GAS = 8.314462618  # J/(mol K)
P_STANDARD = 1.0  # bar


def standard_gibbs(T: torch.Tensor, dH: float, dS: float) -> torch.Tensor:
    """ΔG°_b = ΔH° − T_b ΔS°  (J/mol), with ΔH° and ΔS° taken as temperature-independent.   (B,)"""
    return dH - T * dS


def log_K(T: torch.Tensor, dH: float, dS: float) -> torch.Tensor:
    """ln K_b = −ΔG°_b / (R T_b) = −ΔH°/(R T_b) + ΔS°/R : a straight line in 1/T (van 't Hoff).   (B,)"""
    return -standard_gibbs(T, dH, dS) / (R_GAS * T)


def moles(n0: torch.Tensor, nu: torch.Tensor, xi: torch.Tensor) -> torch.Tensor:
    """n_bi = n⁰_bi + ν_i ξ_b.   n0: (B, S), nu: (S,), xi: (B,) → (B, S).  Nothing summed (outer product)."""
    return n0 + torch.einsum("i,b->bi", nu, xi)


def log_Q(n: torch.Tensor, P: torch.Tensor, nu: torch.Tensor) -> torch.Tensor:
    """ln Q_b = ν_i ln a_bi with a_bi = x_bi P_b / P°.   n: (B, S), P: (B,) → (B,).  i (species) is summed."""
    x = n / n.sum(-1, keepdim=True)  # mole fractions x_bi
    ln_a = torch.log(x) + torch.log(P / P_STANDARD)[:, None]
    return torch.einsum("i,bi->b", nu, ln_a)


def extent_bounds(n0: torch.Tensor, nu: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Allowed range of ξ_b: no amount may go negative.   n0: (B, S) → (ξ_min (B,), ξ_max (B,)).

    reactants (ν < 0) give ξ ≤ n⁰_i / |ν_i|;  products (ν > 0) give ξ ≥ −n⁰_i / ν_i.
    """
    limit = n0 / nu.abs()  # (B, S)
    product = nu > 0
    xi_min = torch.where(product, -limit, -torch.inf).amax(-1)
    xi_max = torch.where(product, torch.inf, limit).amin(-1)
    return xi_min, xi_max


def van_t_hoff_slope(dH: float) -> float:
    """d ln K / d(1/T) = −ΔH° / R : exothermic (ΔH° < 0) ⇒ K falls as T rises, endothermic ⇒ K rises."""
    return -dH / R_GAS


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/physchem/chemical_equilibrium/tensors.py
"""Build the concrete tensors: stoichiometry, starting moles and the batch of conditions (T, P).

Thermodynamic data are approximate standard values at 298 K (gas phase, P° = 1 bar), assumed independent of
temperature. They are good to a few percent near room temperature and only indicative at 700 K.
"""

import torch

DTYPE = torch.float64

REACTIONS = {
    "n2o4": {
        "label": "N₂O₄ ⇌ 2 NO₂",
        "species": ["N₂O₄", "NO₂"],
        "nu": [-1.0, 2.0],  # stoichiometric numbers ν_i (negative = reactant)
        "n0": [1.0, 0.0],  # starting moles
        "dH": 57.2e3,  # ΔH° ≈ +57.2 kJ/mol (endothermic: breaking the N–N bond)
        "dS": 175.8,  # ΔS° ≈ +176 J/(mol K) (one molecule → two)
        "T_curves": [280.0, 300.0, 320.0, 340.0, 360.0],
    },
    "haber": {
        "label": "N₂ + 3 H₂ ⇌ 2 NH₃",
        "species": ["N₂", "H₂", "NH₃"],
        "nu": [-1.0, -3.0, 2.0],
        "n0": [1.0, 3.0, 0.0],  # stoichiometric feed, as in an ammonia plant
        "dH": -92.2e3,  # ΔH° ≈ −92.2 kJ/mol (exothermic)
        "dS": -198.7,  # ΔS° ≈ −199 J/(mol K) (four molecules → two)
        "T_curves": [600.0, 650.0, 700.0, 750.0, 800.0],
    },
}


def reaction(name: str, device: torch.device) -> dict:
    """The reaction's data with ν (S,) and n0 (S,) as tensors."""
    r = dict(REACTIONS[name])
    r["nu"] = torch.tensor(r["nu"], device=device, dtype=DTYPE)
    r["n0"] = torch.tensor(r["n0"], device=device, dtype=DTYPE)
    return r


def pressures(device: torch.device, n: int = 61) -> torch.Tensor:
    """P_p in bar, shape (P,): log-spaced from 0.1 to 100 bar."""
    return torch.logspace(-1, 2, n, device=device, dtype=DTYPE)


def condition_grid(T_list: list[float], P: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Every (temperature, pressure) pair as one flat batch: T_b, P_b of shape (B,) = (len(T_list) · P,)."""
    T = torch.tensor(T_list, device=P.device, dtype=DTYPE)
    T_b, P_b = torch.meshgrid(T, P, indexing="ij")  # (nT, P) each
    return T_b.reshape(-1), P_b.reshape(-1)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/physchem/chemical_equilibrium/solve.py
"""Find the equilibrium extent ξ_b for every condition at once, by bisection on ln Q(ξ) − ln K.

ln Q grows steadily with ξ (more products, fewer reactants), from −∞ at ξ_min to +∞ at ξ_max, so there is exactly
one root in between and halving the bracket is guaranteed to find it: 60 halvings shrink it by 2⁻⁶⁰ ≈ 10⁻¹⁸.
"""

import torch

from problems._common.progress import steps

from . import equation


def equilibrium_extent(
    n0: torch.Tensor, nu: torch.Tensor, P: torch.Tensor, lnK: torch.Tensor, iterations: int = 60
) -> tuple[torch.Tensor, list[float]]:
    """Return ξ_b (B,) with Q_b = K_b, and the largest |ln Q − ln K| after each halving (for the log).

    n0: (B, S) starting moles, nu: (S,), P: (B,) bar, lnK: (B,).
    """
    lo, hi = equation.extent_bounds(n0, nu)  # (B,), (B,)
    history = []
    for _ in steps(iterations, "bisection"):
        mid = 0.5 * (lo + hi)
        f = equation.log_Q(equation.moles(n0, nu, mid), P, nu) - lnK  # (B,) > 0: too far towards products
        hi = torch.where(f > 0, mid, hi)
        lo = torch.where(f > 0, lo, mid)
        history.append(f.abs().max().item())
    return 0.5 * (lo + hi), history


def add_product_and_reequilibrate(
    n_eq: torch.Tensor, nu: torch.Tensor, P: torch.Tensor, lnK: torch.Tensor, extra: float
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Le Chatelier test: add `extra` mol of the product (last species) to an equilibrium mixture n_eq (B, S).

    Returns (moles just after adding (B, S), ln Q − ln K just after (B,), new equilibrium moles (B, S)).
    """
    n_add = n_eq.clone()
    n_add[:, -1] += extra
    gap = equation.log_Q(n_add, P, nu) - lnK  # > 0: Q > K, so the reaction must run backwards
    xi, _ = equilibrium_extent(n_add, nu, P, lnK)  # extent measured from the new starting point
    return n_add, gap, equation.moles(n_add, nu, xi)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/physchem/chemical_equilibrium/plot.py
"""Animate a pressure sweep: equilibrium conversion vs P, the mixture's composition, van 't Hoff, added product."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

SPECIES_COLORS = ["#2563eb", "#9ca3af", "#dc2626"]  # first reactant, second reactant, product


def animate(
    r: dict,
    T_list: list[float],
    P: torch.Tensor,
    conversion: torch.Tensor,
    x_eq: torch.Tensor,
    T_line: torch.Tensor,
    lnK_line: torch.Tensor,
    P_added: float,
    added: tuple[torch.Tensor, torch.Tensor, torch.Tensor],
    path: Path,
) -> Path:
    """conversion (nT, P) and x_eq (nT, P, S); the last temperature row is the chosen T (drawn in black)."""
    P, conv, x_eq = P.cpu().numpy(), conversion.cpu().numpy(), x_eq.cpu().numpy()
    T_line, lnK_line = T_line.cpu().numpy(), lnK_line.cpu().numpy()
    added = [n.cpu().numpy() for n in added]
    species = r["species"]
    S = len(species)
    sp_colors = [SPECIES_COLORS[0]] + SPECIES_COLORS[1 : S - 1] + [SPECIES_COLORS[-1]]
    n_curves = len(T_list) - 1
    colors = plt.get_cmap("plasma")(np.linspace(0.0, 0.85, n_curves))

    fig = plt.figure(figsize=(8.6, 4.5))
    grid = fig.add_gridspec(3, 2, width_ratios=[1.25, 1], height_ratios=[1, 1, 1.1])
    ax_p = fig.add_subplot(grid[:, 0])
    ax_x = fig.add_subplot(grid[0, 1])
    ax_v = fig.add_subplot(grid[1, 1])
    ax_a = fig.add_subplot(grid[2, 1])

    # --- conversion against pressure, one curve per temperature ---
    for t in range(n_curves):
        if abs(T_list[t] - T_list[-1]) < 1e-9:
            continue  # same as the chosen temperature, drawn in black below
        ax_p.plot(P, conv[t], lw=1.3, color=colors[t], label=f"{T_list[t]:.0f} K")
    ax_p.plot(P, conv[-1], lw=2.4, color="#111827", label=f"{T_list[-1]:.0f} K (chosen)")
    ax_p.axvline(P_added, color="#9ca3af", ls="--", lw=1)
    (dot,) = ax_p.plot([], [], "o", ms=8, color="#f59e0b", mec="#b45309", zorder=4)
    ax_p.set_xscale("log")
    ax_p.set_xlim(P[0], P[-1])
    ax_p.set_ylim(0, 1.02)
    ax_p.set_xlabel("total pressure P (bar)")
    ax_p.set_ylabel(r"conversion  $\xi\,/\,\xi_{\max}$")
    ax_p.set_title(r["label"], fontsize=11)
    ax_p.legend(fontsize=7, loc="best")
    p_text = ax_p.text(0.03, 0.04, "", transform=ax_p.transAxes, fontsize=9)

    # --- composition of the mixture at the current pressure (chosen T) ---
    bars = ax_x.bar(range(S), x_eq[-1, 0], color=sp_colors, width=0.6)
    ax_x.set_xticks(range(S), species)
    ax_x.set_ylim(0, 1.2)  # room for the value labels
    ax_x.set_ylabel("mole fraction", fontsize=8)
    ax_x.set_title(f"mixture at {T_list[-1]:.0f} K", fontsize=9, pad=2)
    ax_x.tick_params(labelsize=8)
    labels = [ax_x.text(i, 0, "", ha="center", va="bottom", fontsize=8) for i in range(S)]

    # --- van 't Hoff plot: ln K against 1000/T ---
    ax_v.plot(1000 / T_line, lnK_line, color="#374151", lw=1.4)
    lnK_pts = np.interp(T_list, T_line, lnK_line)
    ax_v.scatter(1000 / np.array(T_list[:-1]), lnK_pts[:-1], c=colors, s=18, zorder=3)
    ax_v.plot([1000 / T_list[-1]], [lnK_pts[-1]], "*", ms=10, color="#111827")
    ax_v.set_xlabel("1000 / T (1/K)", fontsize=8, labelpad=1)
    ax_v.set_ylabel(r"$\ln K$", fontsize=8)
    ax_v.tick_params(labelsize=7)
    slope = -r["dH"] / 8.314462618
    ax_v.set_title(f"van 't Hoff: slope $-\\Delta H°/R$ = {slope:.0f} K", fontsize=9, pad=2)

    # --- added product, grouped by stage: equilibrium → just after adding → new equilibrium ---
    width = 0.8 / S
    for i in range(S):
        stage_moles = [n[i] for n in added]
        ax_a.bar(np.arange(3) + (i - (S - 1) / 2) * width, stage_moles, width, color=sp_colors[i], label=species[i])
    ax_a.set_xticks(range(3), ["equilibrium", f"+ {species[-1]}", "new equil."])
    ax_a.set_ylabel("moles", fontsize=8)
    ax_a.tick_params(labelsize=8)
    ax_a.legend(fontsize=7, ncol=S, loc="upper left")
    ax_a.set_ylim(0, 1.3 * max(n.max() for n in added))
    ax_a.set_title(f"Le Chatelier: add product at {P_added:.3g} bar", fontsize=9, pad=2)

    def update(step: int):
        dot.set_data([P[step]], [conv[-1, step]])
        p_text.set_text(f"P = {P[step]:.3g} bar   conversion = {conv[-1, step]:.3f}")
        for i in range(S):
            bars[i].set_height(x_eq[-1, step, i])
            labels[i].set_position((i, x_eq[-1, step, i] + 0.02))
            labels[i].set_text(f"{x_eq[-1, step, i]:.2f}")

    fig.tight_layout(pad=0.6, h_pad=0.5, w_pad=1.0)
    return save_gif(fig, update, frame_indices(len(P), 61), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'reaction': 'n2o4',
    'T': 320.0,
    'P': 1.0,
    'add_product': 0.5
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.physchem.chemical_equilibrium")]:
    del sys.modules[name]
main = importlib.import_module("problems.physchem.chemical_equilibrium.main")
problem_dir = Path("problems/physchem/chemical_equilibrium")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
